# 04 · Mediation: how much runs through education?

Add contemporary (2023) education to the baseline on a single common sample and
watch the persistence coefficient attenuate. The attenuation is the share of the
effect mediated by human capital (~9% in the paper). **Manuscript: Table 4.**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "pct_highschool_or_more_(2023)": "pct_hs_2023",
    "Pop 2010": "pop_2010",
    "Established firms 1989": "firms_1989",
    "Established firms 2022": "firms_2022",
    "income_1989_real_2023": "income_1989_real",
})

for c in ["Index_v2", "income_1989_real", "pct_hs_1990", "pct_hs_2023",
          "pop_2010", "firms_1989", "firms_2022"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# guard the logs
for v in ["firms_2022", "pop_2010", "firms_1989"]:
    df.loc[df[v] <= 0, v] = np.nan

df["log_firms_2022"] = np.log1p(df["firms_2022"])
df["log_pop_2010"] = np.log(df["pop_2010"])
df["income_1989_std"] = (df["income_1989_real"] - df["income_1989_real"].mean()) / df["income_1989_real"].std()
df["pct_hs_1990_std"] = (df["pct_hs_1990"] - df["pct_hs_1990"].mean()) / df["pct_hs_1990"].std()

# one common sample for both models so the comparison is apples-to-apples
keep = ["log_firms_2022", "Index_v2", "income_1989_std", "pct_hs_1990_std",
        "log_pop_2010", "pct_hs_2023", "State"]
d = df[keep].replace([np.inf, -np.inf], np.nan).dropna()
vc = d["State"].value_counts()
d = d[d["State"].isin(vc[vc > 1].index)]   # keep states estimable under C(State)
print(f"Common N = {len(d)}, States = {d['State'].nunique()}")

baseline = smf.ols(
    "log_firms_2022 ~ Index_v2 + income_1989_std + pct_hs_1990_std "
    "+ log_pop_2010 + C(State)", data=d).fit(cov_type="HC3")
print("\n--- Baseline ---"); print(baseline.summary())

mediation = smf.ols(
    "log_firms_2022 ~ Index_v2 + income_1989_std + pct_hs_1990_std "
    "+ log_pop_2010 + pct_hs_2023 + C(State)", data=d).fit(cov_type="HC3")
print("\n--- + Education 2023 ---"); print(mediation.summary())

b0, b1 = baseline.params["Index_v2"], mediation.params["Index_v2"]
print(f"\nPersistence: {b0:.4f} -> {b1:.4f}  (attenuation {100*(b0-b1)/b0:.1f}%)")